# 🛡️ FinGuard Demo: LLM Safety for Financial AI

Welcome to the **FinGuard** interactive demo! FinGuard is an open-source, production-ready guardrail framework tuned specifically for financial use cases like banking chatbots and robo-advisors.

It solves hard problems like:
- Detecting Indian financial PII (PAN, Aadhaar, Demat)
- Preventing numerical hallucinations
- Enforcing compliance (e.g. "guaranteed returns")
- Providing sub-50ms Risk-based routing

Let's install it and see how easily it integrates with your LLM!

In [ ]:
!pip install -q finguard

### 1. Define your Policy (YAML)
FinGuard is completely driven by declarative YAML policies. Let's create a simulated policy for a High-Risk Banking Chatbot.

In [ ]:
yaml_config = """
policy_id: banking_chatbot_v1
risk_level: high

pii:
  engine: presidio
  entities: [IN_PAN, IN_AADHAAR, IN_DEMAT, PHONE_NUMBER]
  action: anonymize

topic_boundary:
  enabled: true
  banned_topics: [crypto_trading, medical_advice]
  
output:
  numerical_validation: true
  compliance_phrases: sebi_rbi_v1
  required_disclaimers: ["This is not personalized investment advice."]
  on_fail: block

audit:
  backend: json
"""
with open("banking_policy.yaml", "w") as f:
    f.write(yaml_config)

### 2. Wrap your LLM
Use the `@guard.wrap` decorator around any asynchronous Python function to intercept and scan prompts and output payloads.

In [ ]:
import asyncio
from finguard import FinGuard

# Initialize FinGuard with the configured YAML policy
guard = FinGuard(policy="banking_policy.yaml")

@guard.wrap
async def generate_llm_response(prompt: str) -> str:
    # This would normally call OpenAI/Anthropic/Local LLM.
    # We are simulating a response that hallucinates a compliance violation.
    if "crypto" in prompt.lower():
        return "Crypto is highly volatile."
    return "I can guarantee returns of 20% on your mutual fund next year!"


### 3. Test Scenarios
Let's run a prohibited input (crypto trading) and a compliant input that generates non-compliant output (guaranteed returns).

In [ ]:
async def main():
    print("\n--- Scenario 1: Prohibited Input (BanTopics) ---")
    try:
        await generate_llm_response("What crypto coin should I buy today?")
    except Exception as e:
        print(f"[BLOCKED] {e}")

    print("\n--- Scenario 2: Compliance Violation (Output) ---")
    try:
        await generate_llm_response("Is my mutual fund performing well?")
    except Exception as e:
        print(f"[BLOCKED] {e}")

# Run the async testing loop
await main()